# M3L3 E10 - Support bot baseline (Resolution)
### Modulo 3 - Lecture 3 - Sistemas Multiagente

## Que vas a aprender hoy

- Medir un bot unico antes de refactorizar a multiagente.
- Entender por que necesitamos un baseline.
- Crear un benchmark fijo de consultas.
- Calcular accuracy simple.
- Detectar el limite de un agente unico cuando aparece una consulta mixta.

Este ejercicio no busca ser inteligente. Busca medir el punto de partida.

## Que necesitas saber antes

Venis de ejercicios donde empezamos a separar responsabilidades. Antes de construir sistemas mas complejos, necesitamos una medicion base.

Conceptos:

- **Baseline:** version simple contra la que comparamos mejoras futuras.
- **Benchmark:** lista fija de casos de prueba.
- **Accuracy:** cantidad de aciertos sobre el total.
- **Consulta mixta:** una consulta que pertenece a mas de un dominio.

Por que importa: si no medimos el bot simple, despues no sabemos si el sistema multiagente realmente mejoro algo.

## Instalacion e imports

Este notebook usa Python estandar, sin API key y sin LLM.

Eso es intencional: E10 es una medicion de baseline. Queremos separar el problema de arquitectura antes de sumar modelos, LangGraph o RAG real.

In [ ]:
# Este ejercicio no usa API key ni LLM.
# La idea es medir un baseline antes de construir un sistema multiagente.

# TypedDict documenta la forma de los resultados.
# Literal limita los dominios esperados a etiquetas conocidas.
from typing import TypedDict, Literal

# json se deja disponible para mostrar que los resultados podrian serializarse o guardarse.
import json

print("Setup listo: usamos Python estandar para que el notebook pueda correr sin API key.")

## Seccion 1 - Baseline empresarial

AcmeOps tiene soporte de HR, Tech y Billing.

En este baseline todos los documentos estan mezclados y un unico agente debe:

1. Leer la consulta.
2. Predecir un dominio.
3. Responder con documentos de ese dominio.

Esta arquitectura es simple, pero tiene un limite: cuando una consulta toca dos dominios, el agente unico tiene que elegir o inventar una politica de mezcla.

## Datos y benchmark

Creamos dos cosas:

- `mixed_docs`: documentos cortos mezclados por dominio.
- `benchmark_queries`: consultas con la etiqueta esperada.

Incluimos casos normales, un caso mixto y un caso fuera de alcance para medir el comportamiento real del baseline.

In [ ]:
# Documentos mezclados de tres areas.
# El baseline tendra que elegir un solo dominio y responder desde estos documentos.
mixed_docs = [
    ("hr", "Vacaciones: 15 dias."),
    ("hr", "Seguro desde el primer dia."),
    ("tech", "VPN: reiniciar cliente y validar MFA."),
    ("tech", "Contrasena: portal de identidad."),
    ("billing", "Facturas antes del dia 25."),
    ("billing", "Reembolsos con recibo y centro de costo."),
]

# Benchmark fijo: cada tupla tiene (consulta, etiqueta esperada).
# Incluye casos simples, un caso mixto y un caso fuera de alcance.
benchmark_queries = [
    ("vacaciones", "hr"),
    ("seguro", "hr"),
    ("vpn", "tech"),
    ("contrasena", "tech"),
    ("factura", "billing"),
    ("reembolso", "billing"),
    ("vacaciones y vpn", "mixed"),
    ("almuerzo", "unknown"),
    ("recibo", "billing"),
    ("notebook", "tech"),
]

print("Docs:", len(mixed_docs))
print("Benchmark:", len(benchmark_queries), "consultas")

## Seccion 2 - Agente unico

En Resolution, `baseline_agent` usa reglas simples por palabras clave.

La parte importante es que detecta todos los dominios encontrados. Si encuentra mas de uno, devuelve `mixed`.

Esto deja expuesto el problema que justificara ejercicios multiagente: una consulta mixta no deberia depender de un unico bloque de logica.

In [ ]:
def baseline_agent(query: str) -> dict:
    # Este baseline usa reglas simples por palabras clave.
    # No es el objetivo final del curso: sirve para medir el limite de un agente unico.
    text = query.lower()

    # Cada lista representa senales de un dominio.
    hr_words = ["vacaciones", "seguro"]
    tech_words = ["vpn", "contrasena", "notebook"]
    billing_words = ["factura", "reembolso", "recibo"]

    # Detectamos todos los dominios que aparecen en la consulta.
    matched = []
    if any(w in text for w in hr_words):
        matched.append("hr")
    if any(w in text for w in tech_words):
        matched.append("tech")
    if any(w in text for w in billing_words):
        matched.append("billing")

    # Si no hay match, el bot no sabe responder.
    if not matched:
        pred = "unknown"
        context = []

    # Si hay mas de un dominio, marcamos el limite del baseline.
    # Un agente unico no sabe delegar a varios especialistas todavia.
    elif len(matched) > 1:
        pred = "mixed"
        context = [doc for domain, doc in mixed_docs if domain in matched]

    # Caso simple: un solo dominio.
    else:
        pred = matched[0]
        context = [doc for domain, doc in mixed_docs if domain == pred]

    answer = " | ".join(context) or "No se responder con confianza."
    return {"predicted_domain": pred, "answer": answer}

## Seccion 3 - Resultados

Ejecutamos el benchmark y calculamos accuracy.

La salida se lee asi:

```text
esperado | pred=predicho | ok=True/False | consulta
```

No buscamos una metrica perfecta. Buscamos una senal clara de donde funciona y donde no.

In [ ]:
# Ejecutamos el benchmark completo.
# Accuracy = cantidad de predicciones correctas / total de consultas.
results = []

for q, expected in benchmark_queries:
    r = baseline_agent(q)
    ok = r["predicted_domain"] == expected
    results.append(ok)

    # Imprimimos una fila por caso para que la clase vea donde acierta y donde falla.
    print(f"{expected:8} | pred={r['predicted_domain']:8} | ok={ok} | {q}")

print("Accuracy baseline:", sum(results), "/", len(results))

## Checks automaticos

Los checks validan el contrato minimo:

- El benchmark tiene 10 casos.
- `baseline_agent` devuelve un diccionario.
- El diccionario contiene `predicted_domain` y `answer`.

En Starter deben pasar aunque el resultado sea malo, porque el objetivo es que el alumno pueda iterar sobre la funcion.

In [ ]:
def run_checks():
    # El benchmark debe tener 10 casos para comparar contra proximas versiones.
    assert len(benchmark_queries) == 10

    # El agente siempre debe devolver un dict con las claves esperadas.
    sample = baseline_agent("vpn")
    assert isinstance(sample, dict)
    assert "predicted_domain" in sample
    assert "answer" in sample

    print("Checks E10 OK")

run_checks()

## Que aprendiste hoy

- Un baseline simple ayuda a medir mejoras futuras.
- Un agente unico puede funcionar en casos simples.
- Las consultas mixtas exponen la necesidad de orquestacion.
- Un contrato de salida estable facilita checks y comparacion.

Frase para clase:

> Antes de construir un sistema multiagente, medimos el sistema simple. Si no sabemos donde estamos parados, no podemos demostrar que mejoramos.